#### P3-05
# Feature Engineering
#### M03

## 1. Setup

In [1]:
import pandas as pd
from google.colab import drive
from sklearn.feature_selection import mutual_info_classif

drive.mount('/content/drive')
pd.set_option('display.max_columns', None)

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks"
TARGET = 'is_canceled'
MIN_CATEGORY_COUNT = 100

Mounted at /content/drive


## 2. Load Data

In [2]:
hotel_train = pd.read_csv(f"{DATA_DIR}/hotel_bookings_train.csv")
hotel_val = pd.read_csv(f"{DATA_DIR}/hotel_bookings_val.csv")
hotel_test = pd.read_csv(f"{DATA_DIR}/hotel_bookings_test.csv")

print(hotel_train.shape)
print(hotel_val.shape)
print(hotel_test.shape)

(58990, 37)
(10544, 37)
(17406, 37)


Preview of the raw training data. The two `Unnamed` columns are leftover index columns from saving and reloading CSV files.

In [3]:
hotel_train.head(5)

,Unnamed: 0.1,Unnamed: 0,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date_month_num,arrival_date,booking_date
0,0,0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2015-07-01,7,2015-07-01,2014-07-24
1,1,1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2015-07-01,7,2015-07-01,2013-06-24
2,2,2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,0.0,0.0,0,Transient,75.0,0,0,Check-Out,2015-07-02,7,2015-07-01,2015-06-24
3,3,3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,0.0,0,Transient,75.0,0,0,Check-Out,2015-07-02,7,2015-07-01,2015-06-18
4,4,4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,0.0,0,Transient,98.0,0,1,Check-Out,2015-07-03,7,2015-07-01,2015-06-17


## 3. Feature Engineering

### 3.1 Fixed-Rule Features and Column Removal


In [4]:
DROP_COLUMNS = [
    'reservation_status', 'reservation_status_date',
    'assigned_room_type',
    'arrival_date', 'booking_date',
    'arrival_date_year', 'arrival_date_week_number', 'arrival_date_month_num',
    'Unnamed: 0', 'Unnamed: 0.1',
]


def get_booking_history(row):
    if row['previous_cancellations'] == 0 and row['previous_bookings_not_canceled'] == 0:
        return 'No Previous History'
    elif row['previous_cancellations'] > 0 and row['previous_bookings_not_canceled'] == 0:
        return 'Previous Cancellations Only'
    elif row['previous_cancellations'] == 0 and row['previous_bookings_not_canceled'] > 0:
        return 'Previous Successful Bookings'
    else:
        return 'Both'


def add_basic_features(d):
    d = d.drop(columns=DROP_COLUMNS, errors='ignore')
    d['total_nights'] = d['stays_in_weekend_nights'] + d['stays_in_week_nights']
    d['booking_history'] = d.apply(get_booking_history, axis=1)
    return d


hotel_train = add_basic_features(hotel_train)
hotel_val = add_basic_features(hotel_val)
hotel_test = add_basic_features(hotel_test)

### 3.2 Rare Category Grouping

The frequent categories are identified from the training set and then applied to all three datasets.

In [5]:
for col in ['agent', 'company']:
    for d in (hotel_train, hotel_val, hotel_test):
        d[col] = d[col].fillna('Unknown').astype(str)


def group_rare_categories(col, min_count=MIN_CATEGORY_COUNT):
    frequent = hotel_train[col].value_counts()
    frequent = frequent[frequent >= min_count].index
    for d in (hotel_train, hotel_val, hotel_test):
        d[col] = d[col].where(d[col].isin(frequent), 'Other')


for col in ['country', 'agent', 'company']:
    group_rare_categories(col)

## 4. Validation Checks

In [6]:
print(hotel_train[['country', 'agent', 'company']].nunique())
print(list(hotel_train.columns) == list(hotel_val.columns) == list(hotel_test.columns))
print(hotel_train.isna().sum().sum(), hotel_val.isna().sum().sum(), hotel_test.isna().sum().sum())
print(hotel_train.dtypes.value_counts())

country    32
agent      51
company     7
dtype: int64
True
0 0 0
int64      15
object     12
float64     2
Name: count, dtype: int64


**Results**

- All three datasets have identical columns and no missing values.
- After grouping, `country` has 32 categories, `agent` 51 and `company` 7. `agent` is on the high side because each category becomes its own column after one-hot encoding. If the model looks noisy, a higher grouping threshold (for example 500) is an option.
- Each dataset has 29 columns: 28 features plus the target `is_canceled`.

## 5. Save Processed Datasets


In [9]:
splits = {
    'train': hotel_train,
    'val': hotel_val,
    'test': hotel_test
}

for name, d in splits.items():
    d.to_csv(f"{DATA_DIR}/hotel_{name}_fe.csv", index=False)

## 7. Summary

**What the processed datasets contain**

- 28 feature columns plus the target `is_canceled`, identical across train, validation and test.
- Leaking, date-related, duplicate and index columns removed.
- Two new features: `total_nights` and `booking_history`.
- `country`, `agent` and `company` treated as categories with rare values grouped into `Other` (threshold learned from train).
